# AI Termika — PINN řešení

Physics-Informed Neural Network pro predikci šíření tepla v kovové tyči.

Rovnice vedení tepla: $\frac{\partial u}{\partial t} = \alpha \cdot \frac{\partial^2 u}{\partial x^2}$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

# nastavení zařízení
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám: {device}")

torch.manual_seed(42)
np.random.seed(42)

## 1. Načtení dat

In [ ]:
DATA_DIR = "ai_termika_dataset/student"

train_df = pd.read_csv(f"{DATA_DIR}/train_measurements.csv")
boundary_df = pd.read_csv(f"{DATA_DIR}/boundary_partial.csv")
initial_df = pd.read_csv(f"{DATA_DIR}/initial_sparse.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_points.csv")

with open(f"{DATA_DIR}/constants.json") as f:
    constants = json.load(f)

alpha = constants["alpha"]
print(f"Přibližné alpha: {alpha}")
print(f"Train bodů: {len(train_df)}, Boundary bodů: {len(boundary_df)}, Initial bodů: {len(initial_df)}")
print(f"Test bodů: {len(test_df)}")

In [ ]:
# Převod dat na tensory
def df_to_tensors(df, cols_x, col_y=None):
    X = torch.tensor(df[cols_x].values, dtype=torch.float32, device=device)
    if col_y:
        y = torch.tensor(df[col_y].values, dtype=torch.float32, device=device).unsqueeze(1)
        return X, y
    return X

X_train, y_train = df_to_tensors(train_df, ["x", "t"], "u")
X_boundary, y_boundary = df_to_tensors(boundary_df, ["x", "t"], "u")
X_initial, y_initial = df_to_tensors(initial_df, ["x", "t"], "u")
X_test = df_to_tensors(test_df, ["x", "t"])

print(f"X_train shape: {X_train.shape}")

## 2. Vizualizace dat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rozmístění datových bodů
ax = axes[0]
ax.scatter(train_df["t"], train_df["x"], c=train_df["u"], cmap="inferno", s=20, label="train")
ax.scatter(boundary_df["t"], boundary_df["x"], c="red", marker="s", s=30, label="boundary")
ax.scatter(initial_df["t"], initial_df["x"], c="green", marker="^", s=30, label="initial")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.set_title("Rozmístění datových bodů")
ax.legend()

# Počáteční podmínka
ax = axes[1]
ax.scatter(initial_df["x"], initial_df["u"], c="green", marker="^", s=50, label="initial")
ax.set_xlabel("x")
ax.set_ylabel("u(x, 0)")
ax.set_title("Počáteční podmínka")
ax.legend()

plt.tight_layout()
plt.show()

## 3. Definice PINN modelu

MLP síť s Tanh aktivacemi — vhodné pro PINN, protože Tanh je nekonečněkrát diferencovatelná.

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden_size=128, n_layers=5):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(2, hidden_size))
        layers.append(nn.Tanh())
        
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_size, 1))
        
        self.net = nn.Sequential(*layers)
        
        # Inicializace vah — Xavier
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)

model = PINN(hidden_size=128, n_layers=5).to(device)
print(model)
print(f"Počet parametrů: {sum(p.numel() for p in model.parameters()):,}")

## 4. Loss funkce pro PINN

Celková loss = L_data + λ₁·L_boundary + λ₂·L_initial + λ₃·L_physics

Physics loss počítá residual rovnice vedení tepla pomocí automatické diferenciace.

In [ ]:
def physics_residual(model, x_col, alpha_val):
    """
    Spočítá residual rovnice vedení tepla:
    r = du/dt - alpha * d²u/dx²
    
    x_col: tensor [N, 2] s requires_grad=True
    """
    x_col = x_col.requires_grad_(True)
    u = model(x_col)
    
    # Gradienty
    grad_u = torch.autograd.grad(
        u, x_col,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    
    du_dx = grad_u[:, 0:1]  # ∂u/∂x
    du_dt = grad_u[:, 1:2]  # ∂u/∂t
    
    # Druhá derivace ∂²u/∂x²
    d2u_dx2 = torch.autograd.grad(
        du_dx, x_col,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True
    )[0][:, 0:1]
    
    residual = du_dt - alpha_val * d2u_dx2
    return residual


def sample_collocation_points(n_points, device):
    """Náhodné kolokační body v doméně [0,1] x [0,1]."""
    x = torch.rand(n_points, 1, device=device)
    t = torch.rand(n_points, 1, device=device)
    return torch.cat([x, t], dim=1)

## 5. Trénování

In [ ]:
# Hyperparametry
N_EPOCHS = 15000
LR = 1e-3
N_COLLOCATION = 2000  # počet kolokačních bodů pro physics loss

# Váhy loss funkcí
LAMBDA_DATA = 1.0
LAMBDA_BOUNDARY = 5.0
LAMBDA_INITIAL = 5.0
LAMBDA_PHYSICS = 0.5

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)

mse = nn.MSELoss()

# Logování
history = {
    "total": [], "data": [], "boundary": [], "initial": [], "physics": []
}

In [ ]:
print("Začínám trénování...")

for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    
    # 1. Data loss — MSE na trénovacích měřeních
    u_pred_train = model(X_train)
    loss_data = mse(u_pred_train, y_train)
    
    # 2. Boundary loss
    u_pred_boundary = model(X_boundary)
    loss_boundary = mse(u_pred_boundary, y_boundary)
    
    # 3. Initial condition loss
    u_pred_initial = model(X_initial)
    loss_initial = mse(u_pred_initial, y_initial)
    
    # 4. Physics loss — residual rovnice vedení tepla
    x_col = sample_collocation_points(N_COLLOCATION, device)
    residual = physics_residual(model, x_col, alpha)
    loss_physics = torch.mean(residual ** 2)
    
    # Celková loss
    loss_total = (
        LAMBDA_DATA * loss_data
        + LAMBDA_BOUNDARY * loss_boundary
        + LAMBDA_INITIAL * loss_initial
        + LAMBDA_PHYSICS * loss_physics
    )
    
    loss_total.backward()
    optimizer.step()
    scheduler.step()
    
    # Logování
    history["total"].append(loss_total.item())
    history["data"].append(loss_data.item())
    history["boundary"].append(loss_boundary.item())
    history["initial"].append(loss_initial.item())
    history["physics"].append(loss_physics.item())
    
    if (epoch + 1) % 1000 == 0:
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1:5d}/{N_EPOCHS} | "
            f"Total: {loss_total.item():.6f} | "
            f"Data: {loss_data.item():.6f} | "
            f"Boundary: {loss_boundary.item():.6f} | "
            f"Initial: {loss_initial.item():.6f} | "
            f"Physics: {loss_physics.item():.6f} | "
            f"LR: {lr_now:.1e}"
        )

print("\nTrénování dokončeno!")

## 6. Průběh trénování

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Celková loss
ax = axes[0]
ax.semilogy(history["total"], label="Total", alpha=0.7)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Celková loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Jednotlivé složky
ax = axes[1]
ax.semilogy(history["data"], label="Data", alpha=0.7)
ax.semilogy(history["boundary"], label="Boundary", alpha=0.7)
ax.semilogy(history["initial"], label="Initial", alpha=0.7)
ax.semilogy(history["physics"], label="Physics", alpha=0.7)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Složky loss funkce")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Vizualizace predikce

In [ ]:
# Predikce na husté mřížce
n_grid = 100
x_grid = np.linspace(0, 1, n_grid)
t_grid = np.linspace(0, 1, n_grid)
X_mesh, T_mesh = np.meshgrid(x_grid, t_grid)

grid_points = torch.tensor(
    np.column_stack([X_mesh.ravel(), T_mesh.ravel()]),
    dtype=torch.float32, device=device
)

model.eval()
with torch.no_grad():
    u_grid = model(grid_points).cpu().numpy().reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmapa predikce
ax = axes[0]
c = ax.pcolormesh(T_mesh, X_mesh, u_grid, shading="auto", cmap="inferno")
plt.colorbar(c, ax=ax, label="u(x,t)")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.set_title("Predikce modelu — u(x,t)")

# Profily v různých časech
ax = axes[1]
for t_val in [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]:
    x_line = torch.tensor(
        np.column_stack([x_grid, np.full_like(x_grid, t_val)]),
        dtype=torch.float32, device=device
    )
    with torch.no_grad():
        u_line = model(x_line).cpu().numpy()
    ax.plot(x_grid, u_line, label=f"t={t_val:.2f}")

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Teplotní profily")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Generování submission.csv

In [ ]:
model.eval()
with torch.no_grad():
    u_pred_test = model(X_test).cpu().numpy().flatten()

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "u": np.round(u_pred_test, 6)
})

submission.to_csv("submission.csv", index=False)
print(f"Uloženo {len(submission)} predikcí do submission.csv")
print(submission.head(10))

In [ ]:
# Kontrola: statistiky predikcí
print(f"Predikce — min: {u_pred_test.min():.4f}, max: {u_pred_test.max():.4f}, mean: {u_pred_test.mean():.4f}")
print(f"Train data — min: {train_df['u'].min():.4f}, max: {train_df['u'].max():.4f}, mean: {train_df['u'].mean():.4f}")